In [1]:
import numpy as np
import pickle, os, time
import spectres
from POSEIDON.core import (create_star, create_planet, define_model,
                            make_atmosphere, compute_spectrum, wl_grid_constant_R)
from POSEIDON.constants import R_J, M_J, R_Sun

with open("opacities_created/opac_eq_chemistry.pickle", "rb") as f: 
    opac = pickle.load(f)
print("Opacity loaded")

target_species = ["H2O", "CH4", "CO", "CO2", "NH3"]
pressure_bar = np.logspace(3, -6, 100)       
wl_min, wl_max, R_highres = 0.4, 10.0, 15000
wl_highres = wl_grid_constant_R(wl_min, wl_max, R_highres)
wl_binned = np.load("equi_chem_data/wl_binned.npy")

model = define_model("equilibrium_model", ["H2", "He"], target_species,
                      PT_profile="file_read", X_profile="file_read")
star = create_star(1.0 * R_Sun, 5778.0, 4.44, 0.0)
print("Model, star, wl grids ready")

Opacity loaded
Model, star, wl grids ready


In [2]:
import json, glob

# equilibrium VMRs from Phase 1 (same planet order as survey folders)
equilibrium_vmr_all = np.load("equilibrium_vmr_all.npy")   # (8426, 44, 100)
print("Equilibrium VMR:", equilibrium_vmr_all.shape)

# params in the SAME order (matched pairs) - from survey folders
BASE_DIR = "freckll_survey_output"
planet_dirs = sorted(glob.glob(f"{BASE_DIR}/planet_*"))
records = []
for pdir in planet_dirs:
    with open(f"{pdir}/params.json") as f:
        records.append(json.load(f))
print(f"Params: {len(records)} planets")

assert len(records) == len(equilibrium_vmr_all), "MISMATCH: params vs VMR count!"

# species indices - from the same species_names used in Phase 1
species_names = np.load("species_names.npy").tolist()
species_idx = {n: species_names.index(n) for n in target_species}
print("Species indices:", species_idx)

Equilibrium VMR: (8426, 44, 100)
Params: 8426 planets
Species indices: {'H2O': 35, 'CH4': 32, 'CO': 34, 'CO2': 27, 'NH3': 31}


In [3]:
def build_X_input(vmr_array):
    n = len(pressure_bar)
    return np.array([
        np.full(n, 0.83), np.full(n, 0.17),
        vmr_array[species_idx["H2O"], :], vmr_array[species_idx["CH4"], :],
        vmr_array[species_idx["CO"], :],  vmr_array[species_idx["CO2"], :],
        vmr_array[species_idx["NH3"], :],
    ])

def generate_spectrum(record, vmr_array):
    R_p = record["Rp_Rj"] * R_J
    M_p = record["Mp_Mj"] * M_J
    planet = create_planet("planet", R_p, mass=M_p, T_eq=record["Teq_K"])
    T_input = np.full(len(pressure_bar), record["Teq_K"])
    X_input = build_X_input(vmr_array)
    atmosphere = make_atmosphere(planet, model, pressure_bar, 10.0, R_p,
                                  T_input=T_input, X_input=X_input)
    if not atmosphere["is_physical"]:
        return None, False
    spec = compute_spectrum(planet, star, model, atmosphere, opac, wl_highres,
                            spectrum_type="transmission")
    return spec, True

In [4]:

spec, ok = generate_spectrum(records[0], equilibrium_vmr_all[0])
if ok:
    print(f"mean={spec.mean():.3e}, std={spec.std():.3e}, std/mean={spec.std()/spec.mean():.2e}")
    print(f"nan={np.any(np.isnan(spec))}")
else:
    print("NOT physical")

mean=2.346e-02, std=2.221e-04, std/mean=9.47e-03
nan=False


In [5]:
OUT_DIR = "ace_equilibrium_spectra_output"
os.makedirs(OUT_DIR, exist_ok=True)

already = set(int(f.split("_")[1].split(".")[0])
              for f in os.listdir(OUT_DIR) if f.startswith("spec_"))
print(f"{len(already)} already done, {len(records)-len(already)} remaining")

t0 = time.time(); n_proc = 0
for i in range(len(records)):
    if i in already:
        continue
    try:
        spec, ok = generate_spectrum(records[i], equilibrium_vmr_all[i])
        if ok and not (np.any(np.isnan(spec)) or np.any(np.isinf(spec))):
            sb = spectres.spectres(wl_binned, wl_highres, spec, verbose=False)
            np.save(os.path.join(OUT_DIR, f"spec_{i:05d}.npy"), sb)
    except Exception:
        pass
    n_proc += 1
    if n_proc % 100 == 0:
        rate = (time.time()-t0)/n_proc
        print(f"{i}/{len(records)} | {rate:.1f}s/planet | ~{(len(records)-i)*rate/60:.0f} min left", flush=True)

print(f"\nDone. {(time.time()-t0)/60:.1f} min")

0 already done, 8426 remaining
99/8426 | 0.6s/planet | ~77 min left
199/8426 | 0.6s/planet | ~77 min left
299/8426 | 0.6s/planet | ~79 min left
399/8426 | 0.6s/planet | ~77 min left
499/8426 | 0.6s/planet | ~77 min left
599/8426 | 0.6s/planet | ~76 min left
699/8426 | 0.6s/planet | ~72 min left
799/8426 | 0.5s/planet | ~69 min left
899/8426 | 0.5s/planet | ~68 min left
999/8426 | 0.5s/planet | ~67 min left
1099/8426 | 0.5s/planet | ~66 min left
1199/8426 | 0.5s/planet | ~65 min left
1299/8426 | 0.5s/planet | ~64 min left
1399/8426 | 0.5s/planet | ~62 min left
1499/8426 | 0.5s/planet | ~61 min left
1599/8426 | 0.5s/planet | ~60 min left
1699/8426 | 0.5s/planet | ~60 min left
1799/8426 | 0.5s/planet | ~60 min left
1899/8426 | 0.5s/planet | ~60 min left
1999/8426 | 0.6s/planet | ~59 min left
2099/8426 | 0.5s/planet | ~58 min left
2199/8426 | 0.6s/planet | ~57 min left
2299/8426 | 0.6s/planet | ~56 min left
2399/8426 | 0.6s/planet | ~55 min left
2499/8426 | 0.5s/planet | ~54 min left
2599/